In [ ]:
!pip install -q presidio_image_redactor
!python -m spacy download en_core_web_lg

In [ ]:
!sudo apt install tesseract-ocr
!sudo apt install libtesseract-dev

In [ ]:
import inspect
from presidio_image_redactor import DicomImagePiiVerifyEngine

file_path = inspect.getfile(DicomImagePiiVerifyEngine)
file_path

In [3]:
!rm -rf {file_path}
!cp ./dicom_image_pii_verify_engine.py /usr/local/lib/python3.11/dist-packages/presidio_image_redactor/

In [4]:
# RESTART SESSION!!!!

In [1]:
def calculate_precision(gt, pred):
    tp = [i for i in pred if i in gt]
    try:
        precision = len(tp) / len(pred)
    except ZeroDivisionError:
        precision = 0

    return precision

def calculate_recall(gt, pred):
    tp = [i for i in pred if i in gt]
    try:
        recall = len(tp) / len(gt)
    except ZeroDivisionError:
        recall = 0

    return recall

In [ ]:
import json

import glob
import os
import time
import pydicom
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
from presidio_image_redactor import DicomImageRedactorEngine
from presidio_image_redactor import DicomImagePiiVerifyEngine

dicom_engine = DicomImageRedactorEngine()
dicom_image_engine = DicomImagePiiVerifyEngine()

In [3]:
!python prepare_data.py

In [4]:
input_path = "./dicom_files/"
output_parent_dir = "./presidio_result/"

start = time.time()

dicom_engine.redact_from_directory(
    input_dicom_path = input_path,
    output_dir = output_parent_dir,
    fill="contrast",
    save_bboxes=True
)

time_taken = round(time.time() - start, 2)

Output written to presidio_result/dicom_files


In [ ]:
# Original DICOM images
p = Path(input_path).glob("**/*.dcm")
original_files = [x for x in p if x.is_file()]
print(f"Total Original Files {len(original_files)}")

# Redacted DICOM images
p = Path(output_parent_dir).glob("**/*.dcm")
redacted_files = [x for x in p if x.is_file()]
print(f"Total Redacted Files {len(redacted_files)}")

In [ ]:
print(f"Time Taken : {round(time_taken, 2)}")
print(f"Average Time Taken : {round( time_taken / len(redacted_files), 2)}")

In [7]:
# Set paths
data_dir = "./dicom_files/"
gt_path = "./gt_main.json"

with open(gt_path) as json_file:
    gt = json.load(json_file)

# Get list of files
gt_dicom_files = list(gt.keys())

In [ ]:
collect_eval_results = {}

for item in gt_dicom_files:
    gt_file_of_interest = gt[item]

    instance = pydicom.dcmread(os.path.join(data_dir, item))

    verify_image, ocr_results, analyzer_results = dicom_image_engine.verify_dicom_instance(instance)
    _, eval_results = dicom_image_engine.eval_dicom_instance(instance, gt_file_of_interest, use_metadata=True)

    collect_eval_results[item] = eval_results

In [ ]:
total_precision = 0.0
total_recall = 0.0

collect_results = {}

for key, val in collect_eval_results.items():
  presidio_pred = [i["label"].strip().lower() for i in val["all_positives"]]
  gt_pred = [i["label"].strip().lower() for i in val["ground_truth"]]

  precision = calculate_precision(gt_pred, presidio_pred)
  recall = calculate_recall(gt_pred, presidio_pred)

  collect_results[key] = {"gt" : gt_pred, "pred" : presidio_pred, "precision" : precision, "recall" : recall}

  total_precision += precision
  total_recall += recall

final_precision = round(total_precision / len(collect_eval_results), 3)
final_recall = round(total_recall / len(collect_eval_results), 3)
f1_score = round(2 * ((final_precision * final_recall) / (final_precision + final_recall)), 3 )

print(f"Precision : {final_precision}")
print(f"Recall : {final_recall}")
print(f"F1-Score : {f1_score}")

In [ ]:
with open("./results/ner_result/presidio_result.json", "w") as file_out:
  json.dump(collect_results, file_out, indent=4)

In [ ]:
os.makedirs("./image_result", exist_ok=True)
os.makedirs("./results/deid_image_result/presidio/", exist_ok=True)

In [ ]:
presidio_dicom_result = glob.glob("./presidio_result/dicom_files/*.dcm")

for dicom_file in presidio_dicom_result:
    base_name = os.path.basename(dicom_file).replace(".dcm", ".jpg")
    ds = pydicom.dcmread(dicom_file)

    image = Image.fromarray(ds.pixel_array).convert("L")

    file_out = os.path.join("./results/deid_image_result/presidio/", base_name)
    image.save(file_out)